In [1]:
!lscpu | grep -i "avx2"

Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm mpx avx512f avx512dq rdseed adx smap clflushopt clwb avx512cd avx512bw avx512vl xsaveopt xsavec xgetbv1 xsaves arat md_clear arch_capabilities


In [2]:
%%writefile test_avx2.cpp
#include <immintrin.h>
#include <iostream>

int main() {
    alignas(32) double a[4] = {1.5, 2.5, 3.5, 4.5};
    alignas(32) double b[4] = {1.0, 2.0, 3.0, 4.0};
    alignas(32) double c[4];

    __m256d va = _mm256_load_pd(a);
    __m256d vb = _mm256_load_pd(b);
    __m256d vc = _mm256_add_pd(va, vb);
    _mm256_store_pd(c, vc);

    std::cout << "AVX2 Test Result: ";
    for(int i = 0; i < 4; ++i) std::cout << c[i] << " ";
    std::cout << "\n[SUCCESS] Environment Verified." << std::endl;
    return 0;
}

Writing test_avx2.cpp


In [3]:
!g++ -O3 -mavx2 test_avx2.cpp -o test_avx2 && ./test_avx2

AVX2 Test Result: 2.5 4.5 6.5 8.5 
[SUCCESS] Environment Verified.


In [4]:
%%writefile gbm_avx.cpp

#include "avx_mathfun.h"

#include <cmath>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <random>
#include <chrono>

typedef __m256 v8sf;

// ── GBM path generator (AVX float, block RNG) ──────────────────────────────
//
// Parameters:
//   S0, mu, sigma, T  – standard GBM parameters
//   steps, paths      – simulation dimensions
//   data              – output buffer: (steps+1) * num_paths_aligned floats,
//                       32-byte aligned (use std::aligned_alloc)
//   rng_block         – pre-filled normal random buffer: steps * num_paths_aligned
//                       floats, 32-byte aligned, laid out as [step][path]
//
void generate_gbm_paths_avx_float(float S0, float mu, float sigma, float T,
                                  int steps, int paths,
                                  float* data, const float* rng_block)
{
    // Round up to the next multiple of 8 so every row is 32-byte aligned
    int num_paths_aligned = (paths + 7) & ~7;

    // ── scalar constants ────────────────────────────────────────────────────
    float delta_t        = T / static_cast<float>(steps);
    float volatility_drag = (mu - 0.5f * sigma * sigma) * delta_t;
    float normal_scalar  = sigma * std::sqrt(delta_t);

    // ── broadcast into 8-wide AVX registers ────────────────────────────────
    v8sf v_vol_drag   = _mm256_set1_ps(volatility_drag);
    v8sf v_norm_scale = _mm256_set1_ps(normal_scalar);
    v8sf v_s0         = _mm256_set1_ps(S0);

    // ── step 0: every path starts at S0 ────────────────────────────────────
    for (int i = 0; i < num_paths_aligned; i += 8) {
        _mm256_store_ps(&data[0 * num_paths_aligned + i], v_s0);
    }

    // ── time-step loop ──────────────────────────────────────────────────────
    for (int j = 1; j <= steps; j++) {
        for (int i = 0; i < num_paths_aligned; i += 8) {

            // Load 8 pre-generated normals directly – no sampling here
            v8sf v_rand = _mm256_load_ps(&rng_block[(j - 1) * num_paths_aligned + i]);

            // Load previous prices
            v8sf v_prev_price = _mm256_load_ps(&data[(j - 1) * num_paths_aligned + i]);

            // exponent = vol_drag + norm_scale * rand  (FMA)
            v8sf v_exponent = _mm256_fmadd_ps(v_norm_scale, v_rand, v_vol_drag);

            // S_next = S_prev * exp(exponent)  – 8 lanes in parallel
            v8sf v_next_price = _mm256_mul_ps(v_prev_price, exp256_ps(v_exponent));

            // Store results
            _mm256_store_ps(&data[j * num_paths_aligned + i], v_next_price);
        }
    }
}


double price_asian_call_avx(const float* data, float K, float r, float T, int steps, int paths) {
    int num_paths_aligned = (paths + 7) & ~7;

    v8sf v_K = _mm256_set1_ps(K);
    v8sf v_steps = _mm256_set1_ps(static_cast<float>(steps));
    v8sf v_zero = _mm256_setzero_ps();

    // Accumulate final sum in double to prevent precision loss across 100k+ paths
    double total_payoff = 0.0;

    // Process 8 paths simultaneously
    for (int i = 0; i < num_paths_aligned; i += 8) {
        v8sf v_sum = _mm256_setzero_ps();

        // Sum prices from step 1 to steps (standard Asian average excludes S0)
        for (int j = 1; j <= steps; j++) {
            v8sf v_price = _mm256_load_ps(&data[j * num_paths_aligned + i]);
            v_sum = _mm256_add_ps(v_sum, v_price);
        }

        // average = total / steps
        v8sf v_avg = _mm256_div_ps(v_sum, v_steps);

        // payoff = max(average - K, 0.0)
        v8sf v_payoff = _mm256_max_ps(_mm256_sub_ps(v_avg, v_K), v_zero);

        // Store the 8 parallel payoffs to a temporary array
        alignas(32) float payoffs[8];
        _mm256_store_ps(payoffs, v_payoff);

        // Add to total, respecting the actual path count (ignoring padded lanes)
        int valid_lanes = std::min(8, paths - i);
        for (int k = 0; k < valid_lanes; k++) {
            total_payoff += payoffs[k];
        }
    }

    return (total_payoff / paths) * std::exp(-r * T);
}

// ── AVX Down-and-Out Barrier Call Pricer ────────────────────────────────────
double price_barrier_out_call_avx(const float* data, float K, float B, float r, float T, int steps, int paths) {
    int num_paths_aligned = (paths + 7) & ~7;

    v8sf v_K = _mm256_set1_ps(K);
    v8sf v_B = _mm256_set1_ps(B);
    v8sf v_zero = _mm256_setzero_ps();

    double total_payoff = 0.0;

    // Process 8 paths simultaneously
    for (int i = 0; i < num_paths_aligned; i += 8) {
        // Initialize a mask to all 1s (true).
        // Comparing zero == zero is a safe trick to generate an all-1s bitmask.
        v8sf v_valid = _mm256_cmp_ps(v_zero, v_zero, _CMP_EQ_OQ);

        // Check barrier condition from step 0 to steps
        for (int j = 0; j <= steps; j++) {
            v8sf v_price = _mm256_load_ps(&data[j * num_paths_aligned + i]);

            // Compare: true (all 1s) if price >= B, false (all 0s) if price < B
            v8sf v_cmp = _mm256_cmp_ps(v_price, v_B, _CMP_GE_OQ);

            // Bitwise AND accumulates knocked-out states. Once a lane hits 0, it stays 0.
            v_valid = _mm256_and_ps(v_valid, v_cmp);
        }

        // Calculate final raw payoff: max(S_T - K, 0.0)
        v8sf v_final_price = _mm256_load_ps(&data[steps * num_paths_aligned + i]);
        v8sf v_payoff = _mm256_max_ps(_mm256_sub_ps(v_final_price, v_K), v_zero);

        // Apply validity mask. Knocked out paths will have their payoffs bitwise-ANDed with 0.
        v_payoff = _mm256_and_ps(v_payoff, v_valid);

        alignas(32) float payoffs[8];
        _mm256_store_ps(payoffs, v_payoff);

        int valid_lanes = std::min(8, paths - i);
        for (int k = 0; k < valid_lanes; k++) {
            total_payoff += payoffs[k];
        }
    }

    return (total_payoff / paths) * std::exp(-r * T);
}

int main() {

    // ── 1. Simulation parameters ────────────────────────────────────────────
    const float S0    = 100.0f;
    const float mu    = 0.05f;
    const float sigma = 0.20f;
    const float T     = 1.0f;
    const int   steps = 252;
    const int   paths = 1'000'000;

    // Pricing parameters
    const float K     = 100.0f;
    const float B     = 90.0f;
    const float r     = 0.05f;

    int num_paths_aligned = (paths + 7) & ~7;

    // ── 2. Aligned allocation ───────────────────────────────────────────────
    const size_t data_bytes = static_cast<size_t>(steps + 1) * num_paths_aligned * sizeof(float);
    const size_t rng_bytes  = static_cast<size_t>(steps)     * num_paths_aligned * sizeof(float);

    float* data = static_cast<float*>(std::aligned_alloc(32, data_bytes));
    float* rng  = static_cast<float*>(std::aligned_alloc(32, rng_bytes));

    if (!data || !rng) {
        std::cerr << "aligned_alloc failed – not enough memory\n";
        std::free(data);
        std::free(rng);
        return 1;
    }

    // ── 3. Block RNG fill (timed separately) ────────────────────────────────
    std::mt19937 gen(42);
    std::normal_distribution<float> dist(0.0f, 1.0f);

    auto t_rng_start = std::chrono::high_resolution_clock::now();

    const size_t total_randoms = static_cast<size_t>(steps) * num_paths_aligned;
    for (size_t k = 0; k < total_randoms; ++k) {
        rng[k] = dist(gen);
    }

    auto t_rng_end = std::chrono::high_resolution_clock::now();
    double rng_ms = std::chrono::duration<double, std::milli>(t_rng_end - t_rng_start).count();

    // ── 4. Run simulation (timed separately) ────────────────────────────────
    auto t_sim_start = std::chrono::high_resolution_clock::now();

    generate_gbm_paths_avx_float(S0, mu, sigma, T, steps, paths, data, rng);

    auto t_sim_end = std::chrono::high_resolution_clock::now();
    double sim_ms = std::chrono::duration<double, std::milli>(t_sim_end - t_sim_start).count();

    // ── 5. Asian Call Pricing (timed separately) ────────────────────────────
    auto t_asian_start = std::chrono::high_resolution_clock::now();

    double asian_price = price_asian_call_avx(data, K, r, T, steps, paths);

    auto t_asian_end = std::chrono::high_resolution_clock::now();
    double asian_ms = std::chrono::duration<double, std::milli>(t_asian_end - t_asian_start).count();

    // ── 6. Barrier Knock-Out Pricing (timed separately) ─────────────────────
    auto t_barrier_start = std::chrono::high_resolution_clock::now();

    double barrier_price = price_barrier_out_call_avx(data, K, B, r, T, steps, paths);

    auto t_barrier_end = std::chrono::high_resolution_clock::now();
    double barrier_ms = std::chrono::duration<double, std::milli>(t_barrier_end - t_barrier_start).count();

    // ── 7. Timing report & Output ───────────────────────────────────────────
    double total_ms = rng_ms + sim_ms + asian_ms + barrier_ms;

    std::cout << std::fixed << std::setprecision(4);
    std::cout << "\n=== Results =====================================\n";
    std::cout << "  Asian Call Price        : $" << asian_price << "\n";
    std::cout << "  Barrier Knock-Out Price : $" << barrier_price << "\n";

    std::cout << "\n=== Benchmark (" << paths << " paths) ==============\n";
    std::cout << "  RNG fill time           : " << std::setw(8) << rng_ms     << " ms\n";
    std::cout << "  GBM Simulation time     : " << std::setw(8) << sim_ms     << " ms\n";
    std::cout << "  Asian Pricing time      : " << std::setw(8) << asian_ms   << " ms\n";
    std::cout << "  Barrier Pricing time    : " << std::setw(8) << barrier_ms << " ms\n";
    std::cout << "  -----------------------------------------------\n";
    std::cout << "  Total Execution Time    : " << std::setw(8) << total_ms   << " ms\n";
    std::cout << "=================================================\n";

    std::free(data);
    std::free(rng);
    return 0;
}

Writing gbm_avx.cpp


In [17]:
!g++ -O3 -march=native -mfma -std=c++17 -o gbm_avx gbm_avx.cpp && ./gbm_avx


=== Results =====================================
  Asian Call Price        : $5.7751
  Barrier Knock-Out Price : $8.9052

=== Benchmark (1000000 paths) ==============
  RNG fill time           : 4725.4211 ms
  GBM Simulation time     : 698.8097 ms
  Asian Pricing time      : 178.5874 ms
  Barrier Pricing time    : 198.3339 ms
  -----------------------------------------------
  Total Execution Time    : 5801.1521 ms


In [6]:
ls

gbm_avx.cpp  sample_data/  test_avx2*  test_avx2.cpp


In [7]:
%%writefile test.cu

#include <stdio.h>

// Kernel that sets every element to a given value
__global__ void initS0(float* devA, float value) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    devA[idx] = value;
}

int main() {
    const int paths = 1024;

    // 1. Allocate host array and initialize to 0.0f
    float* A = nullptr;
    cudaMallocHost(&A, paths * sizeof(float));
    for (int i = 0; i < paths; i++) A[i] = 0.0f;

    // 2. Allocate device array
    float* devA = nullptr;
    cudaMalloc(&devA, paths * sizeof(float));

    // 3. Launch kernel: 1 block, 1024 threads
    initS0<<<1, paths>>>(devA, 100.0f);
    cudaDeviceSynchronize();

    // 4. Copy results back to host
    cudaMemcpy(A, devA, paths * sizeof(float), cudaMemcpyDeviceToHost);

    // 5. Print first 5 values to verify
    printf("First 5 values:\n");
    for (int i = 0; i < 5; i++) {
        printf("A[%d] = %.2f\n", i, A[i]);
    }

    cudaFree(devA);
    cudaFreeHost(A);
    return 0;
}

Writing test.cu


In [8]:
!nvcc test.cu -o test && ./test

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
First 5 values:
A[0] = 100.00
A[1] = 100.00
A[2] = 100.00
A[3] = 100.00
A[4] = 100.00


In [9]:
%%writefile test2.cu

#include <stdio.h>
#include <curand_kernel.h>

// Kernel to initialize one curandState per thread
__global__ void setup_kernel(curandState* states, unsigned long long seed) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    // Each thread gets the same seed but a unique sequence number (idx),
    // guaranteeing independent streams of random numbers.
    curand_init(seed, idx, 0, &states[idx]);
}

// Kernel to generate one standard-normal sample per thread
__global__ void test_rng(curandState* states, float* devA) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    // Load this thread's RNG state, draw one N(0,1) sample, store it back
    curandState localState = states[idx];
    devA[idx] = curand_normal(&localState);
    states[idx] = localState;   // write updated state back to global memory
}

int main() {
    const int paths            = 1'000'000;
    const int threads_per_block = 256;
    const int blocks_per_grid  = (paths + threads_per_block - 1) / threads_per_block;

    printf("paths=%d  threads_per_block=%d  blocks_per_grid=%d\n\n",
           paths, threads_per_block, blocks_per_grid);

    // --- Host output array (pinned for fast transfers) ---
    float* A = nullptr;
    cudaMallocHost(&A, paths * sizeof(float));

    // --- Device arrays ---
    float*       devA    = nullptr;
    curandState* devStates = nullptr;

    cudaMalloc(&devA,     paths * sizeof(float));
    cudaMalloc(&devStates, paths * sizeof(curandState));

    // --- Initialise RNG states (seed = 42, sequence = idx, offset = 0) ---
    setup_kernel<<<blocks_per_grid, threads_per_block>>>(devStates, 42ULL);
    cudaDeviceSynchronize();

    // --- Generate one N(0,1) variate per path ---
    test_rng<<<blocks_per_grid, threads_per_block>>>(devStates, devA);
    cudaDeviceSynchronize();

    // --- Copy results back and print first 5 ---
    cudaMemcpy(A, devA, paths * sizeof(float), cudaMemcpyDeviceToHost);

    printf("First 5 random N(0,1) values:\n");
    for (int i = 0; i < 5; i++)
        printf("  A[%d] = %+.6f\n", i, A[i]);

    // --- Cleanup ---
    cudaFree(devStates);
    cudaFree(devA);
    cudaFreeHost(A);
    return 0;
}

Writing test2.cu


In [11]:
!nvcc test.cu -o test2 -lcurand && ./test2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
First 5 values:
A[0] = 100.00
A[1] = 100.00
A[2] = 100.00
A[3] = 100.00
A[4] = 100.00


In [27]:
%%writefile fusion.cu

#include <stdio.h>
#include <math.h>
#include <curand_kernel.h>
#include <chrono>
#include <iostream>
#include <iomanip>

// ---------------------------------------------------------------------------
// RNG setup kernel
// ---------------------------------------------------------------------------
__global__ void setup_kernel(curandState* states, unsigned long long seed) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    curand_init(seed, idx, 0, &states[idx]);
}

// ---------------------------------------------------------------------------
// Fused GBM kernel
// ---------------------------------------------------------------------------
__global__ void gbm_kernel_fused(
    float  S0,   float r, float sigma, float T,
    int    steps, int paths,
    curandState* devStates,
    float  B,    float K,
    float* devAsianPayoffs,
    float* devBarrierPayoffs,
    float* devFinalPrices)
{
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    if (idx >= paths) return;

    curandState localState = devStates[idx];

    float dt        = T / (float)steps;
    float drift     = (r - 0.5f * sigma * sigma) * dt;
    float diffusion = sigma * sqrtf(dt);

    float  currentPrice = S0;
    double runningSum   = 0.0;

    bool isKnockedOut = (S0 < B);

    for (int j = 1; j <= steps; j++) {
        float Z      = curand_normal(&localState);
        currentPrice = currentPrice * expf(drift + diffusion * Z);
        runningSum  += (double)currentPrice;
        if (currentPrice < B) isKnockedOut = true;
    }

    devStates[idx] = localState;

    float avgPrice = (float)(runningSum / steps);
    devAsianPayoffs[idx]   = fmaxf(avgPrice - K, 0.0f);
    devBarrierPayoffs[idx] = isKnockedOut ? 0.0f : fmaxf(currentPrice - K, 0.0f);
    devFinalPrices[idx]    = currentPrice;
}

static double dmean(const float* arr, int n) {
    double sum = 0.0;
    for (int i = 0; i < n; i++) sum += (double)arr[i];
    return sum / n;
}

int main() {
    const float S0    = 100.0f;
    const float K     = 100.0f;
    const float B     =  90.0f;
    const float r     =   0.05f;
    const float sigma =   0.20f;
    const float T     =   1.0f;
    const int   steps =  252;
    const int   paths = 1000000;

    const int threads_per_block = 256;
    const int blocks_per_grid   = (paths + threads_per_block - 1) / threads_per_block;

    // Device allocations
    curandState* devStates         = nullptr;
    float* devAsianPayoffs   = nullptr;
    float* devBarrierPayoffs = nullptr;
    float* devFinalPrices    = nullptr;

    cudaMalloc(&devStates,         paths * sizeof(curandState));
    cudaMalloc(&devAsianPayoffs,   paths * sizeof(float));
    cudaMalloc(&devBarrierPayoffs, paths * sizeof(float));
    cudaMalloc(&devFinalPrices,    paths * sizeof(float));

    float *hAsianPayoffs, *hBarrierPayoffs, *hFinalPrices;
    cudaMallocHost(&hAsianPayoffs,   paths * sizeof(float));
    cudaMallocHost(&hBarrierPayoffs, paths * sizeof(float));
    cudaMallocHost(&hFinalPrices,    paths * sizeof(float));

    // --- Start Timing ---
    auto t_start = std::chrono::high_resolution_clock::now();

    // 1. RNG Setup
    auto t1 = std::chrono::high_resolution_clock::now();
    setup_kernel<<<blocks_per_grid, threads_per_block>>>(devStates, 42ULL);
    cudaDeviceSynchronize();
    auto t2 = std::chrono::high_resolution_clock::now();

    // 2. Simulation Kernel
    auto t3 = std::chrono::high_resolution_clock::now();
    gbm_kernel_fused<<<blocks_per_grid, threads_per_block>>>(
        S0, r, sigma, T, steps, paths,
        devStates, B, K,
        devAsianPayoffs, devBarrierPayoffs, devFinalPrices);
    cudaDeviceSynchronize();
    auto t4 = std::chrono::high_resolution_clock::now();

    // 3. Memory Transfer (D2H)
    auto t5 = std::chrono::high_resolution_clock::now();
    cudaMemcpy(hAsianPayoffs,   devAsianPayoffs,   paths * sizeof(float), cudaMemcpyDeviceToHost);
    cudaMemcpy(hBarrierPayoffs, devBarrierPayoffs, paths * sizeof(float), cudaMemcpyDeviceToHost);
    cudaMemcpy(hFinalPrices,    devFinalPrices,    paths * sizeof(float), cudaMemcpyDeviceToHost);
    auto t6 = std::chrono::high_resolution_clock::now();

    // 4. Host Reduction & Discounting
    auto t7 = std::chrono::high_resolution_clock::now();
    double discount       = exp(-(double)r * (double)T);
    double asian_price    = dmean(hAsianPayoffs,   paths) * discount;
    double barrier_price  = dmean(hBarrierPayoffs, paths) * discount;
    double mean_final     = dmean(hFinalPrices,    paths);
    auto t8 = std::chrono::high_resolution_clock::now();

    auto t_end = std::chrono::high_resolution_clock::now();

    // Calculations for Report
    double rng_ms     = std::chrono::duration<double, std::milli>(t2 - t1).count();
    double sim_ms     = std::chrono::duration<double, std::milli>(t4 - t3).count();
    double memcpy_ms  = std::chrono::duration<double, std::milli>(t6 - t5).count();
    double reduce_ms  = std::chrono::duration<double, std::milli>(t8 - t7).count();
    double total_ms   = std::chrono::duration<double, std::milli>(t_end - t_start).count();

    // --- Output ---
    std::cout << std::fixed << std::setprecision(4);
    std::cout << "\n=== Results (CUDA Fused) =======================\n";
    std::cout << "  Asian Call Price        : $" << asian_price << "\n";
    std::cout << "  Barrier Knock-Out Price : $" << barrier_price << "\n";
    std::cout << "  Simulated Mean S_T      :  " << mean_final << "\n";

    std::cout << "\n=== Benchmark (" << paths << " paths) ==============\n";
    std::cout << "  RNG Setup time          : " << std::setw(8) << rng_ms    << " ms\n";
    std::cout << "  Fused Simulation time   : " << std::setw(8) << sim_ms    << " ms\n";
    std::cout << "  Memcpy (D2H) time       : " << std::setw(8) << memcpy_ms << " ms\n";
    std::cout << "  Host Reduction time     : " << std::setw(8) << reduce_ms << " ms\n";
    std::cout << "  -----------------------------------------------\n";
    std::cout << "  Total Execution Time    : " << std::setw(8) << total_ms  << " ms\n";
    std::cout << "=================================================\n";

    // Cleanup
    cudaFree(devStates); cudaFree(devAsianPayoffs); cudaFree(devBarrierPayoffs); cudaFree(devFinalPrices);
    cudaFreeHost(hAsianPayoffs); cudaFreeHost(hBarrierPayoffs); cudaFreeHost(hFinalPrices);
    return 0;
}

Overwriting fusion.cu


In [28]:
!nvcc fusion.cu -o fusion -lcurand -lm && ./fusion

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

=== Results (CUDA Fused) =======================
  Asian Call Price        : $5.7898
  Barrier Knock-Out Price : $8.9348
  Simulated Mean S_T      :  105.1616

=== Benchmark (1000000 paths) ==============
  RNG Setup time          :  36.0674 ms
  Fused Simulation time   :   8.0176 ms
  Memcpy (D2H) time       :   0.9596 ms
  Host Reduction time     :   9.3382 ms
  -----------------------------------------------
  Total Execution Time    :  54.3831 ms


In [31]:
%%writefile thrust.cu
#include <stdio.h>
#include <math.h>
#include <curand_kernel.h>
#include <thrust/reduce.h>
#include <thrust/device_ptr.h>
#include <thrust/execution_policy.h> // Added for explicit policy
#include <chrono>
#include <iostream>
#include <iomanip>

// ---------------------------------------------------------------------------
// RNG setup kernel
// ---------------------------------------------------------------------------
__global__ void setup_kernel(curandState* states, unsigned long long seed) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    curand_init(seed, idx, 0, &states[idx]);
}

// ---------------------------------------------------------------------------
// Fused GBM kernel
// ---------------------------------------------------------------------------
__global__ void gbm_kernel_fused(
    float  S0,   float r, float sigma, float T,
    int    steps, int paths,
    curandState* devStates,
    float  B,    float K,
    float* devAsianPayoffs,
    float* devBarrierPayoffs,
    float* devFinalPrices)
{
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    if (idx >= paths) return;

    curandState localState = devStates[idx];

    float dt        = T / (float)steps;
    float drift     = (r - 0.5f * sigma * sigma) * dt;
    float diffusion = sigma * sqrtf(dt);

    float  currentPrice = S0;
    double runningSum   = 0.0;
    bool   isKnockedOut = (S0 < B);

    for (int j = 1; j <= steps; j++) {
        float Z      = curand_normal(&localState);
        currentPrice = currentPrice * expf(drift + diffusion * Z);
        runningSum  += (double)currentPrice;
        if (currentPrice < B) isKnockedOut = true;
    }

    devStates[idx] = localState;

    float avgPrice = (float)(runningSum / (double)steps);
    devAsianPayoffs[idx]   = fmaxf(avgPrice - K, 0.0f);
    devBarrierPayoffs[idx] = isKnockedOut ? 0.0f : fmaxf(currentPrice - K, 0.0f);
    devFinalPrices[idx]    = currentPrice;
}

int main() {
    // 1. Check for GPU availability and let the system pick the device
    int deviceCount = 0;
    cudaError_t err = cudaGetDeviceCount(&deviceCount);
    if (err != cudaSuccess || deviceCount == 0) {
        std::cerr << "ERROR: No CUDA devices found! Check if your GPU is enabled.\n";
        return 1;
    }

    const float S0    = 100.0f;
    const float K     = 100.0f;
    const float B     =  90.0f;
    const float r     =   0.05f;
    const float sigma =   0.20f;
    const float T     =   1.0f;
    const int   steps =  252;
    const int   paths = 1000000;

    const int threads_per_block = 256;
    const int blocks_per_grid   = (paths + threads_per_block - 1) / threads_per_block;

    // Device allocations
    curandState* devStates;
    float *devAsian, *devBarrier, *devFinal;

    // Error checking on Mallocs
    if (cudaMalloc(&devStates,  paths * sizeof(curandState)) != cudaSuccess ||
        cudaMalloc(&devAsian,   paths * sizeof(float))       != cudaSuccess ||
        cudaMalloc(&devBarrier, paths * sizeof(float))       != cudaSuccess ||
        cudaMalloc(&devFinal,   paths * sizeof(float))       != cudaSuccess) {
        std::cerr << "ERROR: cudaMalloc failed (possibly out of memory).\n";
        return 1;
    }

    auto t_total_start = std::chrono::high_resolution_clock::now();

    // 2. RNG Setup
    auto t1 = std::chrono::high_resolution_clock::now();
    setup_kernel<<<blocks_per_grid, threads_per_block>>>(devStates, 42ULL);
    cudaDeviceSynchronize();
    auto t2 = std::chrono::high_resolution_clock::now();

    // 3. Simulation Kernel
    auto t3 = std::chrono::high_resolution_clock::now();
    gbm_kernel_fused<<<blocks_per_grid, threads_per_block>>>(
        S0, r, sigma, T, steps, paths,
        devStates, B, K,
        devAsian, devBarrier, devFinal);
    cudaDeviceSynchronize();
    auto t4 = std::chrono::high_resolution_clock::now();

    // 4. Thrust On-Device Reduction (Explicitly using thrust::device)
    auto t5 = std::chrono::high_resolution_clock::now();

    // We can reduce directly using the raw pointers + thrust::device policy
    double asianSum   = thrust::reduce(thrust::device, devAsian,   devAsian   + paths, 0.0, thrust::plus<double>());
    double barrierSum = thrust::reduce(thrust::device, devBarrier, devBarrier + paths, 0.0, thrust::plus<double>());
    double finalSum   = thrust::reduce(thrust::device, devFinal,   devFinal   + paths, 0.0, thrust::plus<double>());

    auto t6 = std::chrono::high_resolution_clock::now();
    auto t_total_end = std::chrono::high_resolution_clock::now();

    // ... (rest of your math/output code remains the same) ...

    // Timing calculations
    double rng_ms    = std::chrono::duration<double, std::milli>(t2 - t1).count();
    double sim_ms    = std::chrono::duration<double, std::milli>(t4 - t3).count();
    double thrust_ms = std::chrono::duration<double, std::milli>(t6 - t5).count();
    double total_ms  = std::chrono::duration<double, std::milli>(t_total_end - t_total_start).count();

    std::cout << std::fixed << std::setprecision(4);
    std::cout << "\n=== Results (Thrust Fused) ===\n";
    std::cout << "  Asian Call Price        : $" << (asianSum / paths * exp(-r*T)) << "\n";
    std::cout << "  Barrier Knock-Out Price : $" << (barrierSum / paths * exp(-r*T)) << "\n";
    std::cout << "\n=== Benchmark (" << paths << " paths) ===\n";
    std::cout << "  RNG Setup: " << rng_ms << " ms | Sim: " << sim_ms << " ms | Thrust: " << thrust_ms << " ms\n";

    cudaFree(devStates); cudaFree(devAsian); cudaFree(devBarrier); cudaFree(devFinal);
    return 0;
}

Overwriting thrust.cu


In [33]:
!nvcc -O3 -arch=sm_75 thrust.cu -o thrust -lcurand && ./thrust


=== Results (Thrust Fused) ===
  Asian Call Price        : $5.7898
  Barrier Knock-Out Price : $8.9348

=== Benchmark (1000000 paths) ===
  RNG Setup: 25.6546 ms | Sim: 7.1617 ms | Thrust: 1.6674 ms
